- Update: 2023.07.04

# 0. 配置参数

- `source`：待查重数据源路径
- `out`：保存结果的路径

In [1]:
source = r'C:\Users\YangTeng\Desktop\查重\Joanne【模板】.xlsx'# 输入带查重文件路径

out = r'C:\Users\YangTeng\Desktop\查重\吉尚重复.xlsx' # 输入输出文件路径

In [2]:
import pandas as pd

In [3]:
# 读取待查重数据
fac_data = pd.read_excel(source,sheet_name='需查重数据',dtype=str).fillna('')

fac_data

,序号,辨识号,OE
0,1,A001,M134479;460107B420;460108B600;460108B620
1,2,A002,M630703;46010JA02ACP;46010JA02A
2,3,A003,M630685;46010ZS00ACP;46010ZS00A
3,4,A004,61195;4601V7
4,5,A005,M630340;4702848020
...,...,...,...
274,275,A275,71916;4601030R20
275,276,A276,PMF441;4601058Y22;4601058Y02
276,277,A277,71921;4601060A01;4601059A11;4601059A01
277,278,A278,71926;4601065C00;4601065C10;4601065C20


## 定义清理OE函数

In [4]:
import re

In [5]:
def normalise_oe(oe):
    
    oe = str(oe).strip()
#     oe = re.sub(r'^[a-zA-Z]*:', '', oe)#去除":"及前面所有字母
    oe = re.sub(r'[\u4e00-\u9fa5]',"",oe) #去除汉字
    oe = re.sub('\.+','',oe)  #去除"."符号
    oe = re.sub('\（','(',oe) #将所有中文"（"符号替换为英文"("
    oe = re.sub('\）',')',oe) #将所有中文"）"符号替换为英文")"
    oe = re.sub('\(.*\)','',oe)  #删除所有"()"及其内部的字符（因为通常不是OE号）
    oe = re.sub('\*+', '', oe) #删除所有的"*"
    oe = re.sub('  +', ';', oe) #将两个以上的空格替换为';'
    oe = re.sub(' ', '', oe) #删除单空格
    oe = re.sub('/+', ';', oe) #将“/"替换为';'
    oe = re.sub('\n', ';', oe) #将换行符替换为';'
    oe = re.sub('[；，,+]', ';', oe) #将[；，,+]替换为';'
    oe = re.sub('\u200e', '', oe)
    oe = re.sub(';$','',oe)
    oe = re.sub('\-', '', oe)

    #list1 = oe26.split(';')
    list1 = [i for i in str(oe).split(';') if i != '' and i.isalpha() == False] #去除空格
    list1 = [re.sub(r'[^0-9]+[:：]','',i) for i in list1]#去除":"及前面所有字母
    list1 = [re.sub(r'[:：]+','',i) for i in list1]#去除":"
    list1 = [i for i in list1 if i !=''and i.isalpha() == False] #去除空格
    
    list2 = list(set(list1))
    list2.sort(key = list1.index)# 去重后保持OE号原来的顺序
    
    oe=';'.join(list2)
    
# #     删除OE号中混杂的年份如'1999';'2018'
    
#     list3 = [i for i in str(oe27).split(';') if len(i) > 3]
    
#     list3 = [i for i in list3 if not (len(i) == 4 and i[0] in ['1', '2'])]

    
#     oe27=';'.join(list3)
    
    return oe.upper()

## 内部查重

In [6]:
fac_data['清理后OE'] = [normalise_oe(fac_data.loc[i, 'OE']) for i in range(len(fac_data))]

fac_data

,序号,辨识号,OE,清理后OE
0,1,A001,M134479;460107B420;460108B600;460108B620,M134479;460107B420;460108B600;460108B620
1,2,A002,M630703;46010JA02ACP;46010JA02A,M630703;46010JA02ACP;46010JA02A
2,3,A003,M630685;46010ZS00ACP;46010ZS00A,M630685;46010ZS00ACP;46010ZS00A
3,4,A004,61195;4601V7,61195;4601V7
4,5,A005,M630340;4702848020,M630340;4702848020
...,...,...,...,...
274,275,A275,71916;4601030R20,71916;4601030R20
275,276,A276,PMF441;4601058Y22;4601058Y02,PMF441;4601058Y22;4601058Y02
276,277,A277,71921;4601060A01;4601059A11;4601059A01,71921;4601060A01;4601059A11;4601059A01
277,278,A278,71926;4601065C00;4601065C10;4601065C20,71926;4601065C00;4601065C10;4601065C20


In [7]:
# 剔除无OE的数据
clean_fac_data = fac_data[fac_data['清理后OE']!='']

clean_fac_data

,序号,辨识号,OE,清理后OE
0,1,A001,M134479;460107B420;460108B600;460108B620,M134479;460107B420;460108B600;460108B620
1,2,A002,M630703;46010JA02ACP;46010JA02A,M630703;46010JA02ACP;46010JA02A
2,3,A003,M630685;46010ZS00ACP;46010ZS00A,M630685;46010ZS00ACP;46010ZS00A
3,4,A004,61195;4601V7,61195;4601V7
4,5,A005,M630340;4702848020,M630340;4702848020
...,...,...,...,...
274,275,A275,71916;4601030R20,71916;4601030R20
275,276,A276,PMF441;4601058Y22;4601058Y02,PMF441;4601058Y22;4601058Y02
276,277,A277,71921;4601060A01;4601059A11;4601059A01,71921;4601060A01;4601059A11;4601059A01
277,278,A278,71926;4601065C00;4601065C10;4601065C20,71926;4601065C00;4601065C10;4601065C20


In [8]:
new_fac_data = clean_fac_data[['辨识号','清理后OE']].copy()
new_fac_data['清理后OE'] = new_fac_data['清理后OE'].str.split(';')
new_fac_data = new_fac_data.explode('清理后OE')

new_fac_data

,辨识号,清理后OE
0,A001,M134479
0,A001,460107B420
0,A001,460108B600
0,A001,460108B620
1,A002,M630703
...,...,...
277,A278,4601065C00
277,A278,4601065C10
277,A278,4601065C20
278,A279,M39975


In [9]:
merge_fac_data = pd.merge(new_fac_data,new_fac_data,on='清理后OE',how='left')

merge_fac_data

,辨识号_x,清理后OE,辨识号_y
0,A001,M134479,A001
1,A001,460107B420,A001
2,A001,460108B600,A001
3,A001,460108B620,A001
4,A002,M630703,A002
...,...,...,...
1087,A278,4601065C00,A278
1088,A278,4601065C10,A278
1089,A278,4601065C20,A278
1090,A279,M39975,A279


In [10]:
group_merge_fac_data = merge_fac_data.groupby(['清理后OE'],as_index=False,sort=False).agg({'辨识号_y':';'.join})

group_merge_fac_data

,清理后OE,辨识号_y
0,M134479,A001
1,460107B420,A001
2,460108B600,A001
3,460108B620,A001
4,M630703,A002
...,...,...
1087,4601065C00,A278
1088,4601065C10,A278
1089,4601065C20,A278
1090,M39975,A279


In [11]:
repeat_fac_data = pd.merge(new_fac_data,group_merge_fac_data,on='清理后OE',how='left').fillna('')

repeat_fac_data

,辨识号,清理后OE,辨识号_y
0,A001,M134479,A001
1,A001,460107B420,A001
2,A001,460108B600,A001
3,A001,460108B620,A001
4,A002,M630703,A002
...,...,...,...
1087,A278,4601065C00,A278
1088,A278,4601065C10,A278
1089,A278,4601065C20,A278
1090,A279,M39975,A279


In [12]:
fac_repeat_data = repeat_fac_data[repeat_fac_data['辨识号']!=repeat_fac_data['辨识号_y']]

fac_repeat_data

,辨识号,清理后OE,辨识号_y


In [13]:
group_fac_repeat_data = fac_repeat_data.groupby(['辨识号'],as_index=False,sort=False).agg({'清理后OE':';'.join, '辨识号_y':';'.join})

group_fac_repeat_data

,辨识号,清理后OE,辨识号_y


In [14]:
group_fac_repeat_data.rename(columns={'清理后OE':'表内重复OE号','辨识号_y':'表内重复辨识号'},inplace=True)

In [15]:
group_fac_repeat_data

,辨识号,表内重复OE号,表内重复辨识号


In [16]:
group_fac_repeat_data['表内重复OE号'] = group_fac_repeat_data['表内重复OE号'].str.split(';').apply(lambda x: sorted(list(set(x)), key=x.index))
group_fac_repeat_data['表内重复辨识号'] = group_fac_repeat_data['表内重复辨识号'].str.split(';').apply(lambda x: sorted(list(set(x)), key=x.index))

In [17]:
group_fac_repeat_data

,辨识号,表内重复OE号,表内重复辨识号


In [18]:
# for i in range(len(group_fac_repeat_data)):
#     for item in group_fac_repeat_data['表内重复辨识号'][i]:
#         if item == group_fac_repeat_data['辨识号'][i]:
#             print(i)
#             print(group_fac_repeat_data['辨识号'][i])
#             print(item)
#             group_fac_repeat_data['表内重复辨识号'][i].remove(item)

In [19]:
group_fac_repeat_data

,辨识号,表内重复OE号,表内重复辨识号


In [20]:
group_fac_repeat_data['表内重复OE号'] = group_fac_repeat_data['表内重复OE号'].apply(lambda x: ';'.join(x))
group_fac_repeat_data['表内重复辨识号'] = group_fac_repeat_data['表内重复辨识号'].apply(lambda x: ';'.join(x))

In [21]:
group_fac_repeat_data

,辨识号,表内重复OE号,表内重复辨识号


In [22]:
result = pd.merge(fac_data,group_fac_repeat_data,on='辨识号',how='left').fillna('')

result

,序号,辨识号,OE,清理后OE,表内重复OE号,表内重复辨识号
0,1,A001,M134479;460107B420;460108B600;460108B620,M134479;460107B420;460108B600;460108B620,,
1,2,A002,M630703;46010JA02ACP;46010JA02A,M630703;46010JA02ACP;46010JA02A,,
2,3,A003,M630685;46010ZS00ACP;46010ZS00A,M630685;46010ZS00ACP;46010ZS00A,,
3,4,A004,61195;4601V7,61195;4601V7,,
4,5,A005,M630340;4702848020,M630340;4702848020,,
...,...,...,...,...,...,...
274,275,A275,71916;4601030R20,71916;4601030R20,,
275,276,A276,PMF441;4601058Y22;4601058Y02,PMF441;4601058Y22;4601058Y02,,
276,277,A277,71921;4601060A01;4601059A11;4601059A01,71921;4601060A01;4601059A11;4601059A01,,
277,278,A278,71926;4601065C00;4601065C10;4601065C20,71926;4601065C00;4601065C10;4601065C20,,


In [23]:
result[result['清理后OE']=='']

,序号,辨识号,OE,清理后OE,表内重复OE号,表内重复辨识号


## 和PTSKU查重

In [24]:
# 读取系统数据
sku_data = pd.read_excel(source,sheet_name='PTSKU',dtype=str).fillna('')

sku_data

,SKU,OE
0,CSL03489-C,CS103489;UB3941920
1,CSL03715-C,CS103715;0K20141920;0K20141920A;B45541920
2,CSL11956-C,CS11956;J3241867
3,CSL29876-C,CS129876;4171022650;4171022660
4,CSL33739-FC,CS133739;4171028000;4171039001(Frankberg)
...,...,...
1585,BMC00544,36000544;36000758;8603662;36002537
1586,BMC00101,1694300101;A1694300201;1694300301
1587,BMC00020,SJJ500020;LR014527
1588,BMC00001,0064300001;A1684300101;1684300001


In [25]:
sku_data['清理后OE'] = [normalise_oe(sku_data.loc[i, 'OE']) for i in range(len(sku_data))]

sku_data

,SKU,OE,清理后OE
0,CSL03489-C,CS103489;UB3941920,CS103489;UB3941920
1,CSL03715-C,CS103715;0K20141920;0K20141920A;B45541920,CS103715;0K20141920;0K20141920A;B45541920
2,CSL11956-C,CS11956;J3241867,CS11956;J3241867
3,CSL29876-C,CS129876;4171022650;4171022660,CS129876;4171022650;4171022660
4,CSL33739-FC,CS133739;4171028000;4171039001(Frankberg),CS133739;4171028000;4171039001
...,...,...,...
1585,BMC00544,36000544;36000758;8603662;36002537,36000544;36000758;8603662;36002537
1586,BMC00101,1694300101;A1694300201;1694300301,1694300101;A1694300201;1694300301
1587,BMC00020,SJJ500020;LR014527,SJJ500020;LR014527
1588,BMC00001,0064300001;A1684300101;1684300001,0064300001;A1684300101;1684300001


In [26]:
clean_sku_data = sku_data[sku_data['清理后OE']!='']

clean_sku_data

,SKU,OE,清理后OE
0,CSL03489-C,CS103489;UB3941920,CS103489;UB3941920
1,CSL03715-C,CS103715;0K20141920;0K20141920A;B45541920,CS103715;0K20141920;0K20141920A;B45541920
2,CSL11956-C,CS11956;J3241867,CS11956;J3241867
3,CSL29876-C,CS129876;4171022650;4171022660,CS129876;4171022650;4171022660
4,CSL33739-FC,CS133739;4171028000;4171039001(Frankberg),CS133739;4171028000;4171039001
...,...,...,...
1585,BMC00544,36000544;36000758;8603662;36002537,36000544;36000758;8603662;36002537
1586,BMC00101,1694300101;A1694300201;1694300301,1694300101;A1694300201;1694300301
1587,BMC00020,SJJ500020;LR014527,SJJ500020;LR014527
1588,BMC00001,0064300001;A1684300101;1684300001,0064300001;A1684300101;1684300001


In [27]:
new_sku_data = clean_sku_data[['SKU','清理后OE']].copy()
new_sku_data['清理后OE'] = new_sku_data['清理后OE'].str.split(';')
new_sku_data = new_sku_data.explode('清理后OE')

new_sku_data

,SKU,清理后OE
0,CSL03489-C,CS103489
0,CSL03489-C,UB3941920
1,CSL03715-C,CS103715
1,CSL03715-C,0K20141920
1,CSL03715-C,0K20141920A
...,...,...
1587,BMC00020,LR014527
1588,BMC00001,0064300001
1588,BMC00001,A1684300101
1588,BMC00001,1684300001


In [28]:
merge_data = pd.merge(new_fac_data,new_sku_data,on='清理后OE',how='left').fillna('')

merge_data

,辨识号,清理后OE,SKU
0,A001,M134479,
1,A001,460107B420,
2,A001,460108B600,
3,A001,460108B620,
4,A002,M630703,
...,...,...,...
1087,A278,4601065C00,
1088,A278,4601065C10,
1089,A278,4601065C20,
1090,A279,M39975,


In [29]:
repeat_data = merge_data[merge_data['SKU']!='']

repeat_data

,辨识号,清理后OE,SKU
94,A037,CS360033,CSL60033-C


In [30]:
sku_repeat_data = repeat_data.groupby(['辨识号'],as_index=False,sort=False).agg({'清理后OE':';'.join,'SKU':';'.join})

sku_repeat_data

,辨识号,清理后OE,SKU
0,A037,CS360033,CSL60033-C


In [31]:
sku_repeat_data.rename(columns={'清理后OE':'系统重复OE','SKU':'重复PTSKU'},inplace=True)

sku_repeat_data

,辨识号,系统重复OE,重复PTSKU
0,A037,CS360033,CSL60033-C


In [32]:
sku_repeat_data['系统重复OE'] = sku_repeat_data['系统重复OE'].str.split(';').apply(lambda x: sorted(list(set(x)), key=x.index))
sku_repeat_data['重复PTSKU'] = sku_repeat_data['重复PTSKU'].str.split(';').apply(lambda x: sorted(list((set(x))), key=x.index))

In [33]:
sku_repeat_data

,辨识号,系统重复OE,重复PTSKU
0,A037,[CS360033],[CSL60033-C]


In [34]:
merge_data

,辨识号,清理后OE,SKU
0,A001,M134479,
1,A001,460107B420,
2,A001,460108B600,
3,A001,460108B620,
4,A002,M630703,
...,...,...,...
1087,A278,4601065C00,
1088,A278,4601065C10,
1089,A278,4601065C20,
1090,A279,M39975,


In [35]:
sku_repeat_data['系统重复OE'] = sku_repeat_data['系统重复OE'].apply(lambda x: ';'.join(x))
sku_repeat_data['重复PTSKU'] = sku_repeat_data['重复PTSKU'].apply(lambda x: ';'.join(x))

In [36]:
sku_repeat_data

,辨识号,系统重复OE,重复PTSKU
0,A037,CS360033,CSL60033-C


In [37]:
result = pd.merge(result,sku_repeat_data,on='辨识号',how='left').fillna('')

result

,序号,辨识号,OE,清理后OE,表内重复OE号,表内重复辨识号,系统重复OE,重复PTSKU
0,1,A001,M134479;460107B420;460108B600;460108B620,M134479;460107B420;460108B600;460108B620,,,,
1,2,A002,M630703;46010JA02ACP;46010JA02A,M630703;46010JA02ACP;46010JA02A,,,,
2,3,A003,M630685;46010ZS00ACP;46010ZS00A,M630685;46010ZS00ACP;46010ZS00A,,,,
3,4,A004,61195;4601V7,61195;4601V7,,,,
4,5,A005,M630340;4702848020,M630340;4702848020,,,,
...,...,...,...,...,...,...,...,...
274,275,A275,71916;4601030R20,71916;4601030R20,,,,
275,276,A276,PMF441;4601058Y22;4601058Y02,PMF441;4601058Y22;4601058Y02,,,,
276,277,A277,71921;4601060A01;4601059A11;4601059A01,71921;4601060A01;4601059A11;4601059A01,,,,
277,278,A278,71926;4601065C00;4601065C10;4601065C20,71926;4601065C00;4601065C10;4601065C20,,,,


In [38]:
result['是否已开发']=''

In [39]:
result

,序号,辨识号,OE,清理后OE,表内重复OE号,表内重复辨识号,系统重复OE,重复PTSKU,是否已开发
0,1,A001,M134479;460107B420;460108B600;460108B620,M134479;460107B420;460108B600;460108B620,,,,,
1,2,A002,M630703;46010JA02ACP;46010JA02A,M630703;46010JA02ACP;46010JA02A,,,,,
2,3,A003,M630685;46010ZS00ACP;46010ZS00A,M630685;46010ZS00ACP;46010ZS00A,,,,,
3,4,A004,61195;4601V7,61195;4601V7,,,,,
4,5,A005,M630340;4702848020,M630340;4702848020,,,,,
...,...,...,...,...,...,...,...,...,...
274,275,A275,71916;4601030R20,71916;4601030R20,,,,,
275,276,A276,PMF441;4601058Y22;4601058Y02,PMF441;4601058Y22;4601058Y02,,,,,
276,277,A277,71921;4601060A01;4601059A11;4601059A01,71921;4601060A01;4601059A11;4601059A01,,,,,
277,278,A278,71926;4601065C00;4601065C10;4601065C20,71926;4601065C00;4601065C10;4601065C20,,,,,


In [40]:
for i in range(len(result)):
    if result.loc[i,'清理后OE'] != '':
        if result.loc[i,'重复PTSKU']!='':
            result.loc[i,'是否已开发'] = '是'
        else:
            if result.loc[i,'表内重复辨识号']!='':
                result.loc[i,'是否已开发'] = '否，表内重复'
            else:
                result.loc[i,'是否已开发'] = '否'
    else:
        result.loc[i,'是否已开发'] = '无OE号，待进一步确认'

In [41]:
result

,序号,辨识号,OE,清理后OE,表内重复OE号,表内重复辨识号,系统重复OE,重复PTSKU,是否已开发
0,1,A001,M134479;460107B420;460108B600;460108B620,M134479;460107B420;460108B600;460108B620,,,,,否
1,2,A002,M630703;46010JA02ACP;46010JA02A,M630703;46010JA02ACP;46010JA02A,,,,,否
2,3,A003,M630685;46010ZS00ACP;46010ZS00A,M630685;46010ZS00ACP;46010ZS00A,,,,,否
3,4,A004,61195;4601V7,61195;4601V7,,,,,否
4,5,A005,M630340;4702848020,M630340;4702848020,,,,,否
...,...,...,...,...,...,...,...,...,...
274,275,A275,71916;4601030R20,71916;4601030R20,,,,,否
275,276,A276,PMF441;4601058Y22;4601058Y02,PMF441;4601058Y22;4601058Y02,,,,,否
276,277,A277,71921;4601060A01;4601059A11;4601059A01,71921;4601060A01;4601059A11;4601059A01,,,,,否
277,278,A278,71926;4601065C00;4601065C10;4601065C20,71926;4601065C00;4601065C10;4601065C20,,,,,否


In [42]:
result[result['是否已开发']=='无OE号，待进一步确认']

,序号,辨识号,OE,清理后OE,表内重复OE号,表内重复辨识号,系统重复OE,重复PTSKU,是否已开发


In [43]:
import datetime

In [44]:
result.to_excel(out.replace('NOW()', datetime.datetime.now().strftime("%Y%m%d_%H%M%S")), index=False)